# Módulo 7: Observabilidade e Avaliações -- Monitoramento em Produção

![Overview](../shared/img/07.drawio.png)

Ao longo deste laboratório, você tem habilitado o **Tracing** em cada recurso do AgentCore. O Runtime exporta os traces OpenTelemetry desde o Módulo 2, e os recursos de Memória e Gateway tiveram a telemetria ativada nos Módulos 4 e 5.

Neste módulo, faremos a exploração desses dados de telemetria acumulados, a implantação da versão final robusta da Aria (V5) e a configuração de **avaliadores customizados (custom evaluators)** que monitorarão o desempenho do agente de forma contínua.

## Objetivos do Laboratório

- **Exploração de rastros**: Navegar pelo painel de observabilidade GenAI do CloudWatch para auditar as inferências, operações de memória e requests no Gateway.
- **Arquitetura de telemetria**: Como a instrumentação via ADOT e a indexação do X-Ray Transaction Search operam no pipeline.
- **Avaliadores Customizados**: Avaliações do tipo LLM-as-judge para auditar a qualidade das respostas e o rigor no uso das ferramentas MCP.
- **Avaliações contínuas (Online evaluations)**: Monitoramento proativo com taxas de amostragem configuráveis no fluxo de log.

## Catch-up

Certifica a integridade da stack construída nos módulos anteriores.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("07")

## Arquitetura de Observabilidade (Background)

Todo deploy corporativo deste laboratório foi orquestrado em conjunto com o pipeline de tracing. O fluxo configurado foi:

### Tracing do Agente

Em todo deploy automático efetuado pelo script auxiliar:
1. Inclusão da biblioteca nativa `aws-opentelemetry-distro` na imagem Docker.
2. Empacotamento do entrypoint corporativo via `opentelemetry-instrument`, instrumentando nativamente as chamadas HTTP, acesso aos FMs (LLMs) e rotinas das ferramentas MCP.
3. Execução da API `tracingConfiguration={"enabled": True}` injetada no AgentCore Runtime.

Esse nível profundo de instrumentação significa que todas as interações executadas desde o Módulo 2 geraram logs estruturados para a AWS.

### Tracing de Infraestrutura (Nível de Recurso AWS)

Nos Módulos 4 e 5 a habilitação na console acoplou os serviços internos. Isso preencheu nativamente os painéis da AWS com eventos complexos em árvore para cada etapa de recuperação.

### Métricas geradas pelas APIs AWS

A Amazon Bedrock emite sinais sem necessidade de configuração ativa — como latência, instabilidade, erros de throttle ou restrição de hardware (CPU/RAM). Tudo acessível com **zero configuração** via CloudWatch (namespace `Bedrock-AgentCore`).

### Arquitetura do Fluxo de Telemetria

![Trace Flow](../shared/img/trace-flow.drawio.png)

> **Documentation:** [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import utils, deploy_agent

# Gather environment variables from all prior modules
env_vars = {}

memory_config = utils.load_config("memory")
if memory_config:
    env_vars["MEMORY_ID"] = memory_config["memory_id"]

gateway_config = utils.load_config("gateway")
if gateway_config:
    env_vars["GATEWAY_ENDPOINT"] = gateway_config.get("gateway_url", "")

print(f"Environment variables: {list(env_vars.keys())}")
print()

# Deploy V5 (tracing is enabled by default on all deploys)
runtime_config = deploy_agent.deploy(
    agent_dir="agent",
    env_vars=env_vars,
)

## Análise Forense dos Logs (Tracing)

Métricas brutas geradas nas iterações anteriores estão consolidadas no Amazon CloudWatch. Vamos exercitar queries mais complexas forçando a inferência, para validar os nós no painel visual.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

In [ ]:
# Trace 1: Simple calculation (exercises code interpreter)
result = test_agent.invoke("What is the square root of 144?", jwt_token=jwt_token)

In [ ]:
# Trace 2: Task management (exercises Gateway + Cedar policy)
result = test_agent.invoke(
    "Create a task: Review observability traces in CloudWatch",
    jwt_token=jwt_token,
)

In [ ]:
# Trace 3: Memory storage (exercises memory service)
result = test_agent.invoke(
    "Remember that I prefer dark mode for all my applications",
    jwt_token=jwt_token,
)

In [ ]:
# Trace 4: List tasks (exercises Gateway read operation)
result = test_agent.invoke("Show me all my current tasks", jwt_token=jwt_token)

## Acessando o CloudWatch da AWS

No console de gerenciamento, abra o **CloudWatch Console**. Navegue na lateral esquerda até **Sinais de Aplicação (Application Signals)** > **GenAI Observabilidade** > **Bedrock AgentCore**. Analise:

### Aba de Agentes
Contém as invocações totais. Ao expandir o log, a estrutura em 'Waterfall' exibe:
- **Root span**: O nó raiz de requisição HTTP
- **LLM spans**: Chamadas de rede para a Modelagem Básica (FM), listando os tokens gastos por payload e a latência de geração
- **Tool spans**: Invocações seguras nas ferramentas restritas do ecossistema AgentCore
- **Memory spans**: Buscas de vetor semântico na infra do DynamoDB
- **Gateway spans**: Passagem do token JWT para interceptação no Cedar e autorização no Target API

### Aba da Memória
Disseca as rotinas em profundidade técnica:
- `CreateEvent`, `RetrieveMemoryRecords`, `ListMemoryRecords` (Comandos da API CRUD)
- Indexação de dados abstratos
- Merging e mesclagem heurística da inferência consolidada

### Aba das Ferramentas Nativas AWS
Dados das instâncias rodando o Browser headless e a sandbox Python:
- Contador de logs e taxas de falhas na sandbox
- Telemetria de CPU/RAM de cada microVM isolada

### Aba do Gateway
Escrutínio do ambiente Gateway e regras de negócio:
- As rotinas padrão do SDK: `List Tools`, `Call Tool`
- Registros determinísticos gerados pelo Policy Engine (O motivo de um ALLOW ou DENY do Cedar)
- Latência da infraestrutura externa (backend)

### Visões Operacionais Avançadas
- O **Mapa de Serviços (Service Map) do AWS X-Ray**: Mostra a topologia em grafos da arquitetura em runtime real.
- O **Log Insights**: Para varredura usando o motor SQL-like de parsing JSON.

> **Boas Práticas AWS:** Recomenda-se analisar a variação da telemetria desde o Módulo 2 e comparar com as restrições adicionadas no módulo do IAM e Gateway.

## Avaliações Computacionais (Custom Evaluators)

A Observabilidade lhe diz *o que* ocorreu em rede. **Avaliações (Evaluations)** dizem com *que grau de eficiência corporativa* a arquitetura atuou.

O Avaliador funciona delegando o crivo a outra LLM via AWS que irá auditar cegamente as instâncias. Você define os seguintes parâmetros:
- **Escala de Precisão (Rating scale)** (Ex: 1 a 5).
- **Rubricas de Inspeção** -- As regras exatas que a AWS forçará o juiz a ler.
- **Escopo da Avaliação** -- Uma auditoria GLOBAL (Sessão completa) ou uma restrita ao TRACE do loop lógico.

| Construto de Nuvem | Descrição Arquitetural |
|---|---|
| **Avaliador (Evaluator)** | Classe que modela o padrão LLM-as-judge para validar respostas. |
| **Evaluation level** | O `SESSION` audita o log inteiro de chat. O `TRACE` foca na decisão atômica da ferramenta MCP. |
| **Avaliação Online** | Avaliação executada de forma nativa e assíncrona nas requisições reais. |
| **Taxa de Amostragem** | % do tráfego desviado pela AWS para ser avaliado. Mitiga custos abusivos em produção. |

> **Docs**: [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)

### Modelagem de Avaliador 1: Qualidade Global de Sessão (SESSION)

Este avaliador será encarregado de classificar a sessão macro no limite 1 a 5. Invocamos a rede interna via `bedrock-agentcore-control`.

A configuração embarca as seguintes instruções da classe `llmAsAJudge`:
- **Instruções** repletas de marcações de string dinâmicas (placeholders).
- **Range de Medição** amparada por rótulos restritos e descritivos.
- **Model Config** alocando o id da fundation model que agirá como Auditor.

In [ ]:
import boto3
from botocore.exceptions import ClientError
import sys; sys.path.insert(0, '..')
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)

# Evaluator 1: Response Quality (SESSION level)
#
# SESSION evaluators require at least one of these placeholders:
#   {available_tools}, {context}, {actual_tool_trajectory},
#   {expected_tool_trajectory}, {assertions}

response_quality_instructions = """You are evaluating the quality of an AI assistant named Aria.

Here is the context of the conversation:
{context}

Here are the tools available to the assistant:
{available_tools}

Here is the actual tool trajectory:
{actual_tool_trajectory}

Score on this scale:
5 - Excellent: Fully addresses request with accurate, complete information.
4 - Good: Mostly complete and accurate with minor gaps.
3 - Adequate: Partially addresses request with noticeable gaps.
2 - Poor: Fails to adequately address request.
1 - Unacceptable: Wrong, hallucinated, or harmful.

Weigh: Accuracy, Completeness, Relevance, Clarity, Groundedness.
Provide brief justification before the numeric rating."""

print("Rubric for ResponseQuality:")
print(response_quality_instructions[:200] + "...")
print()

try:
    resp = control.create_evaluator(
        evaluatorName="ResponseQuality",
        description="Evaluates helpfulness, accuracy, and completeness of responses",
        level="SESSION",
        evaluatorConfig={
            "llmAsAJudge": {
                "instructions": response_quality_instructions,
                "ratingScale": {
                    "numerical": [
                        {"value": 1, "label": "Unacceptable", "definition": "Wrong, hallucinated, or harmful"},
                        {"value": 2, "label": "Poor", "definition": "Fails to adequately address request"},
                        {"value": 3, "label": "Adequate", "definition": "Partially addresses request with noticeable gaps"},
                        {"value": 4, "label": "Good", "definition": "Mostly complete and accurate with minor gaps"},
                        {"value": 5, "label": "Excellent", "definition": "Fully addresses request with accurate, complete information"},
                    ],
                },
                "modelConfig": {
                    "bedrockEvaluatorModelConfig": {
                        "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                    }
                },
            }
        },
    )
    response_quality_id = resp["evaluatorId"]
    print(f"ResponseQuality evaluator created: {response_quality_id}")
    print(f"Status: {resp['status']}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print(f"ResponseQuality evaluator already exists or validation issue: {e.response['Error']['Message']}")
        response_quality_id = "existing"
    else:
        raise

### Modelagem de Avaliador 2: Uso Profundo de Ferramenta (TRACE)

O Auditor focará inteiramente na eficácia granular da adoção do endpoint. O agente atendeu perfeitamente ao MCP ou desperdiçou a API?

In [ ]:
# Evaluator 2: Tool Usage (TRACE level)
#
# TRACE evaluators require at least one of these placeholders:
#   {context}, {assistant_turn}, {expected_response}

tool_usage_instructions = """You are evaluating how well an AI assistant named Aria uses its tools.

Here is the context of the interaction:
{context}

Here is the assistant's response:
{assistant_turn}

Score on this scale:
5 - Optimal: Exactly the right tools, well-formed inputs, no unnecessary calls.
4 - Good: Correct tools with minor inefficiencies.
3 - Acceptable: Suboptimal but functional tool usage.
2 - Poor: Wrong tool selected or many unnecessary calls.
1 - Critical failure: Essential tools not used or severe misuse.

Weigh: Tool selection, Input quality, Efficiency, Completeness, Error handling.
Provide brief justification before the numeric rating."""

print("Rubric for ToolUsage:")
print(tool_usage_instructions[:200] + "...")
print()

try:
    resp = control.create_evaluator(
        evaluatorName="ToolUsage",
        description="Evaluates tool selection and usage efficiency",
        level="TRACE",
        evaluatorConfig={
            "llmAsAJudge": {
                "instructions": tool_usage_instructions,
                "ratingScale": {
                    "numerical": [
                        {"value": 1, "label": "Critical failure", "definition": "Essential tools not used or severe misuse"},
                        {"value": 2, "label": "Poor", "definition": "Wrong tool selected or many unnecessary calls"},
                        {"value": 3, "label": "Acceptable", "definition": "Suboptimal but functional tool usage"},
                        {"value": 4, "label": "Good", "definition": "Correct tools with minor inefficiencies"},
                        {"value": 5, "label": "Optimal", "definition": "Exactly the right tools, well-formed inputs, no unnecessary calls"},
                    ],
                },
                "modelConfig": {
                    "bedrockEvaluatorModelConfig": {
                        "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                    }
                },
            }
        },
    )
    tool_usage_id = resp["evaluatorId"]
    print(f"ToolUsage evaluator created: {tool_usage_id}")
    print(f"Status: {resp['status']}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print(f"ToolUsage evaluator already exists or validation issue: {e.response['Error']['Message']}")
        tool_usage_id = "existing"
    else:
        raise

## Set up online evaluation

Online evaluations attach evaluators to a live agent with a **sampling rate**. They run asynchronously -- they do not slow down the agent's responses.

Online evaluations require:
- A **data source** pointing to the CloudWatch log group that contains the agent's OTel trace data
- An **execution role** with permissions to read logs and invoke Bedrock models
- **Evaluator references** -- either built-in or custom evaluators

AgentCore provides several built-in evaluators:

| Evaluator | Level | What it measures |
|---|---|---|
| `Builtin.Helpfulness` | TRACE | How helpful the agent's responses are |
| `Builtin.GoalSuccessRate` | SESSION | Whether the agent achieved the user's goal |
| `Builtin.Correctness` | TRACE | Factual accuracy of responses |
| `Builtin.Conciseness` | TRACE | Whether responses are appropriately concise |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | Whether the agent chose the right tools |

Below we use the `create_online_evaluation_config` control-plane API to wire up two built-in evaluators to Aria's trace log group. We set the sampling rate to 100% so that evaluation results appear quickly during the workshop. In production, you would typically use a lower rate (e.g., 10%) to control cost.

In [ ]:
# Set up online evaluation using the boto3 control-plane API directly
runtime_config = utils.load_config("runtime")
runtime_id = runtime_config["runtime_id"]
runtime_name = runtime_config["runtime_name"]

# The Runtime's OTel traces are written to this log group automatically
log_group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"

# The service name matches the pattern: {runtime_name}.DEFAULT
service_name = f"{runtime_name}.DEFAULT"

# The evaluation execution role is pre-provisioned by the workshop CloudFormation template
cfn_outputs = utils.get_all_cfn_outputs()
eval_role_arn = cfn_outputs["EvaluationRoleArn"]

print(f"Log group:      {log_group}")
print(f"Service name:   {service_name}")
print(f"Execution role: {eval_role_arn}")
print()

try:
    config = control.create_online_evaluation_config(
        onlineEvaluationConfigName="aria_quality_monitor",
        description="Monitor Aria response quality — 100% sampling for workshop",
        rule={
            "samplingConfig": {
                "samplingPercentage": 100.0,
            },
        },
        dataSourceConfig={
            "cloudWatchLogs": {
                "logGroupNames": [log_group],
                "serviceNames": [service_name],
            },
        },
        evaluators=[
            {"evaluatorId": "Builtin.GoalSuccessRate"},
            {"evaluatorId": "Builtin.Helpfulness"},
        ],
        evaluationExecutionRoleArn=eval_role_arn,
        enableOnCreate=True,
    )
    print("Online evaluation created!")
    print(f"  Config ID: {config.get('onlineEvaluationConfigId', 'N/A')}")
    print(f"  Status:    {config.get('status', 'N/A')}")
except ClientError as e:
    if "already exists" in str(e) or e.response["Error"]["Code"] == "ConflictException":
        print(f"Online evaluation config already exists: {e.response['Error']['Message']}")
    else:
        raise

In [ ]:
# List all evaluators (built-in and custom)
evaluators = control.list_evaluators()
print(f"{'Evaluator ID':<45} {'Name':<25} {'Level':<10}")
print("-" * 80)
for e in evaluators.get("evaluators", []):
    eid = e.get("evaluatorId", "?")
    name = e.get("evaluatorName", "?")
    level = e.get("level", "?")
    marker = " <-- custom" if not eid.startswith("Builtin.") else ""
    print(f"{eid:<45} {name:<25} {level:<10}{marker}")

### Salvar manifesto localmente

In [ ]:
# Save evaluations config for Module 8
eval_config = {
    "custom_evaluators": {
        "ResponseQuality": response_quality_id if "response_quality_id" in dir() else "existing",
        "ToolUsage": tool_usage_id if "tool_usage_id" in dir() else "existing",
    },
    "builtin_evaluators": ["Builtin.GoalSuccessRate", "Builtin.Helpfulness"],
}
utils.save_config("evaluations", eval_config)
print("Evaluations configuration saved")

---

## Run on-demand evaluation with custom evaluators

The custom evaluators we created (ResponseQuality and ToolUsage) are not wired to online evaluation -- they are designed for **on-demand** use. On-demand evaluation lets you score a specific session by:

1. **Downloading span logs** from CloudWatch for a given session ID
2. **Calling the `evaluate()` API** on the data plane with those spans

This is useful for debugging specific conversations, testing evaluator rubrics, and investigating quality issues.

### Step 1: Invoke the agent and capture the session ID

We need a session ID from a recent invocation. Let's run one now and save it.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

# Invoke with a prompt that exercises tools (good for ToolUsage evaluator)
result = test_agent.invoke(
    "Calculate the compound interest on $5,000 at 6% for 10 years, then create a task to review my investment portfolio",
    jwt_token=jwt_token,
)

eval_session_id = result["session_id"]
print(f"\nSession ID for evaluation: {eval_session_id}")

### Step 2: Download span logs from CloudWatch

The `evaluate()` API requires the raw span logs as input. We query CloudWatch Logs Insights to download all spans for the session.

> **Note:** It can take 1--2 minutes for spans to appear in CloudWatch after an invocation. If the query returns empty results, wait and re-run this cell.

In [ ]:
import boto3, json, time
from datetime import datetime, timedelta

region = utils.get_region()
runtime_config = utils.load_config("runtime")
runtime_id = runtime_config["runtime_id"]

logs_client = boto3.client("logs", region_name=region)

def query_logs(log_group_name, query_string):
    """Run a CloudWatch Logs Insights query and return results."""
    start_time = datetime.now() - timedelta(minutes=60)
    end_time = datetime.now()

    query_id = logs_client.start_query(
        logGroupName=log_group_name,
        startTime=int(start_time.timestamp()),
        endTime=int(end_time.timestamp()),
        queryString=query_string,
    )["queryId"]

    while True:
        result = logs_client.get_query_results(queryId=query_id)
        if result["status"] in ("Complete", "Failed"):
            break
        time.sleep(1)

    if result["status"] == "Failed":
        raise Exception("CloudWatch Logs Insights query failed")
    return result["results"]

def get_session_spans(session_id):
    """Download all span logs for a session from both log groups."""
    query = f"""fields @timestamp, @message
    | filter ispresent(scope.name) and ispresent(attributes.session.id)
    | filter attributes.session.id = "{session_id}"
    | sort @timestamp asc"""

    # Runtime log group
    runtime_log_group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"
    runtime_results = query_logs(runtime_log_group, query)
    print(f"  Runtime spans: {len(runtime_results)}")

    # AWS vended spans log group
    aws_results = query_logs("aws/spans", query)
    print(f"  AWS spans:     {len(aws_results)}")

    # Extract JSON messages
    spans = []
    for row in runtime_results + aws_results:
        for field in row:
            if field["field"] == "@message" and field["value"].strip().startswith("{"):
                spans.append(json.loads(field["value"]))
    
    print(f"  Total spans:   {len(spans)}")
    return spans

print(f"Downloading spans for session: {eval_session_id[:16]}...")
print()
session_spans = get_session_spans(eval_session_id)

if not session_spans:
    print("\n⚠ No spans found yet. Wait 1-2 minutes and re-run this cell.")

### Step 3: Run the custom evaluators

Now we call the `evaluate()` **data plane** API with each custom evaluator. The API sends the span data to the LLM judge, which scores the session according to our rubric.

- **ResponseQuality** (SESSION level) — evaluates the entire conversation
- **ToolUsage** (TRACE level) — evaluates each individual turn where tools were used

In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=region)

# Look up custom evaluator IDs
eval_cfg = utils.load_config("evaluations")
custom_evaluators = eval_cfg.get("custom_evaluators", {})

# Get the actual IDs (they may have been saved as "existing" if created in a prior run)
# In that case, look them up from the control plane
all_evaluators = control.list_evaluators().get("evaluators", [])
evaluator_map = {e["evaluatorName"]: e["evaluatorId"] for e in all_evaluators}

rq_id = evaluator_map.get("ResponseQuality", custom_evaluators.get("ResponseQuality"))
tu_id = evaluator_map.get("ToolUsage", custom_evaluators.get("ToolUsage"))

print(f"ResponseQuality evaluator ID: {rq_id}")
print(f"ToolUsage evaluator ID:       {tu_id}")

def run_evaluation(evaluator_id, evaluator_name, spans):
    """Run on-demand evaluation and print results."""
    print(f"\n{'='*60}")
    print(f"Running {evaluator_name} evaluation...")
    print(f"{'='*60}")

    response = data_client.evaluate(
        evaluatorId=evaluator_id,
        evaluationInput={"sessionSpans": spans},
    )

    for result in response["evaluationResults"]:
        if result.get("errorCode"):
            print(f"\n  ERROR: {result['errorCode']} — {result.get('errorMessage', '')}")
            continue

        score = result.get("value", "N/A")
        label = result.get("label", "N/A")
        explanation = result.get("explanation", "")
        tokens = result.get("tokenUsage", {})
        context = result.get("context", {}).get("spanContext", {})

        print(f"\n  Score: {score} — {label}")
        if context.get("traceId"):
            print(f"  Trace: {context['traceId'][:32]}...")
        print(f"  Tokens: {tokens.get('inputTokens', 0)} in / {tokens.get('outputTokens', 0)} out")
        print(f"\n  Explanation:\n  {explanation[:500]}")

    return response["evaluationResults"]

# Run ResponseQuality (SESSION level)
rq_results = run_evaluation(rq_id, "ResponseQuality", session_spans)

# Run ToolUsage (TRACE level)
tu_results = run_evaluation(tu_id, "ToolUsage", session_spans)

### How on-demand evaluation works

The flow you just ran:

1. **Invoked the agent** — this generated OpenTelemetry spans that were written to CloudWatch Logs
2. **Downloaded spans** — queried CloudWatch Logs Insights for all spans matching the session ID, from both the Runtime log group and the `aws/spans` log group
3. **Called `evaluate()`** — passed the raw span data to the data plane API along with the evaluator ID. The service extracted the relevant context from the spans, filled in the evaluator's template placeholders, and sent everything to the judge model
4. **Received scores** — the LLM judge returned a numerical score, label, and explanation for each evaluation target (one result for SESSION-level, one per trace for TRACE-level)

On-demand evaluation is ideal for:
- **Testing evaluator rubrics** before deploying them as online evaluations
- **Investigating specific sessions** that received low online evaluation scores
- **Comparing evaluator versions** side by side on the same session data

## Notas de Engenharia de Avaliadores (System Design)

### Estruturação Semântica de Placeholders

Cada instrução no motor LLM exige o mapeamento em chaves AWS predefinidas na string (Placeholders), populadas com as variáveis de execução de nuvem nativa.

| Nível Hierárquico | Placeholders Essenciais na Configuração | Melhor uso na AWS |
|---|---|---|
| **Sessão (SESSION)** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | Score Macro e de UX. |
| **Restrito (TRACE)** | `{context}`, `{assistant_turn}`, `{expected_response}` | Depuração de micro inferências. |
| **Específico de Ferramenta (TOOL_CALL)** | Padrões de código AWS *(somente nas built-ins)*. | Qualidade no trigger da chamada da REST API. |

Cláusulas estritas com rótulos `expected_` necessitam fortemente de dados validados (arquivos Golden Dataset ou matriz de resposta esperada) via input da `API`. Elas só rodam em modelo lote/on-demand.

### Rating Scale e Regras de Configuração JSON

A topologia de configuração exige que `ratingScale` tenha suporte completo de range numérico. A classe se acopla em `evaluatorConfig`:

```python
evaluatorConfig={
    "llmAsAJudge": {
        "instructions": "...",  # Must include at least one placeholder
        "ratingScale": {
            "numerical": [
                {"value": 1, "label": "Poor", "definition": "..."},
                {"value": 5, "label": "Excellent", "definition": "..."},
            ]
        },
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
            }
        }
    }
}
```

### Online vs Processamento em Massa On-Demand

| Interface Executável | Como opera na Infraestrutura | Cenário e Aplicabilidade Técnica |
|---|---|---|
| **CloudWatch Sink (Online)** | Filtra amostras (Samples) do fluxo contínuo de inferência log-stream nativo da AWS. | Uso Corporativo de painéis operacionais na Produção. |
| **REST Endpoint (On-demand)** | Avalia lotes de matriz JSON extraídos por SDK (via ID). | Ideal para debug local, pipelines CI/CD em regressões e relatórios executivos atrasados. |

Rotinas baseadas na nuvem ativa (online) obrigatoriamente se utilizam do IAM Role de execução restrita para o permissionamento da invocação e acesso as métricas CloudWatch.

## O último passo da Arquitetura Produtiva

Todos os **9 microsserviços do ecosistema AgentCore** operam juntos nativamente.

1. O ambiente gerenciado (**Runtime**) isolando o contêiner Strands Python.
2. Ambientes de teste herméticos via **Interpretador de Código (Code Interpreter)**.
3. Componentes remotos como as sessões do Navegador via **Browser Tool**.
4. Subsistema unificado LTM & STM em nuvem AWS DynamoDB por intermédio da **Memória (AgentCore Memory)**.
5. Proxy reverso do protocolo standard MCP e Roteamento de Ferramentas providos pela infraestrutura restrita do **Gateway (AgentCore Gateway)**.
6. Governança total da topologia JWT RSA originada em `Cognito` via sub-sistema nativo de Auth (Identidade/Identity).
7. Auditoria estrita em tráfego leste-oeste via **Policy Engine (AgentCore Policy)** e a especificação Cedar.
8. Agregados em grafos de traces OpenTelemetry enviados ativamente ao painel do **CloudWatch Observability**.
9. Inferência de qualidade automática orientada à base sólida provida pelo serviço **Avaliações (AgentCore Evaluations)**.

Neste ponto focal do Módulo 8, faremos os testes automatizados de Integração Sistêmica fim a fim.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("07")

---

**Next up: [Module 8 -- Full Deployment Review](../08-full-deployment/notebook.ipynb)**